In [ ]:
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox

# --- Database Setup ---
conn = sqlite3.connect('pharmacy.db')
cursor = conn.cursor()

# Create admin table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS admin (
        username TEXT PRIMARY KEY,
        password TEXT NOT NULL
    )
''')
cursor.execute("INSERT OR IGNORE INTO admin VALUES (?, ?)", ('admin', 'admin123'))

# Create medicines table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS medicines (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        quantity INTEGER NOT NULL,
        price REAL NOT NULL
    )
''')
conn.commit()

# --- Main Pharmacy Dashboard ---
def launch_main_dashboard():
    root = tk.Tk()
    root.title("Pharmacy Management System")
    root.geometry("900x550")
    root.configure(bg="#f4f6f7")

    style = ttk.Style(root)
    style.theme_use('clam')
    style.configure("Treeview.Heading", font=('Segoe UI', 10, 'bold'), background="#007acc", foreground="white")
    style.configure("Treeview", font=('Segoe UI', 10), rowheight=25)
    style.map('Treeview', background=[('selected', '#d9eaf7')])

    def refresh_table():
        for row in tree.get_children():
            tree.delete(row)
        cursor.execute("SELECT * FROM medicines")
        for row in cursor.fetchall():
            tree.insert('', tk.END, values=row)

    def add_medicine():
        name = entry_name.get()
        qty = entry_qty.get()
        price = entry_price.get()
        if name and qty and price:
            try:
                cursor.execute("INSERT INTO medicines (name, quantity, price) VALUES (?, ?, ?)",
                               (name, int(qty), float(price)))
                conn.commit()
                refresh_table()
                clear_fields()
                messagebox.showinfo("Success", "Medicine added!")
            except:
                messagebox.showerror("Error", "Invalid input.")
        else:
            messagebox.showwarning("Input Error", "Please fill all fields.")

    def update_medicine():
        selected = tree.selection()
        if not selected:
            return
        med_id = tree.item(selected[0])['values'][0]
        name = entry_name.get()
        qty = entry_qty.get()
        price = entry_price.get()
        if name and qty and price:
            try:
                cursor.execute("UPDATE medicines SET name=?, quantity=?, price=? WHERE id=?",
                               (name, int(qty), float(price), med_id))
                conn.commit()
                refresh_table()
                clear_fields()
                messagebox.showinfo("Success", "Medicine updated!")
            except:
                messagebox.showerror("Error", "Invalid input.")
        else:
            messagebox.showwarning("Input Error", "Fill all fields.")

    def delete_medicine():
        selected = tree.selection()
        if not selected:
            return
        med_id = tree.item(selected[0])['values'][0]
        cursor.execute("DELETE FROM medicines WHERE id=?", (med_id,))
        conn.commit()
        refresh_table()
        clear_fields()
        messagebox.showinfo("Deleted", "Medicine deleted.")

    def sell_medicine():
        selected = tree.selection()
        if not selected:
            return
        med_id = tree.item(selected[0])['values'][0]
        qty_to_sell = entry_qty.get()
        if not qty_to_sell:
            messagebox.showwarning("Input Error", "Enter quantity to sell.")
            return
        cursor.execute("SELECT quantity, price FROM medicines WHERE id=?", (med_id,))
        result = cursor.fetchone()
        if result:
            available_qty, price = result
            qty_to_sell = int(qty_to_sell)
            if available_qty >= qty_to_sell:
                new_qty = available_qty - qty_to_sell
                cursor.execute("UPDATE medicines SET quantity=? WHERE id=?", (new_qty, med_id))
                conn.commit()
                refresh_table()
                clear_fields()
                total = qty_to_sell * price
                messagebox.showinfo("Sale Complete", f"Sold {qty_to_sell}. Total: ₹{total}")
            else:
                messagebox.showwarning("Stock Error", "Not enough stock.")

    def clear_fields():
        entry_name.delete(0, tk.END)
        entry_qty.delete(0, tk.END)
        entry_price.delete(0, tk.END)

    def on_row_select(event):
        selected = tree.selection()
        if selected:
            values = tree.item(selected[0])['values']
            entry_name.delete(0, tk.END)
            entry_name.insert(0, values[1])
            entry_qty.delete(0, tk.END)
            entry_qty.insert(0, values[2])
            entry_price.delete(0, tk.END)
            entry_price.insert(0, values[3])

    tk.Label(root, text="Pharmacy Management System", font=("Segoe UI", 20, "bold"),
             fg="#007acc", bg="#f4f6f7").pack(pady=10)

    frame = tk.Frame(root, bg="#f4f6f7")
    frame.pack(pady=10)

    label_font = ("Segoe UI", 11)

    tk.Label(frame, text="Medicine Name", font=label_font, bg="#f4f6f7").grid(row=0, column=0, padx=10, pady=5, sticky='w')
    entry_name = ttk.Entry(frame, width=25)
    entry_name.grid(row=0, column=1, pady=5)

    tk.Label(frame, text="Quantity", font=label_font, bg="#f4f6f7").grid(row=1, column=0, padx=10, pady=5, sticky='w')
    entry_qty = ttk.Entry(frame, width=25)
    entry_qty.grid(row=1, column=1, pady=5)

    tk.Label(frame, text="Price (₹)", font=label_font, bg="#f4f6f7").grid(row=2, column=0, padx=10, pady=5, sticky='w')
    entry_price = ttk.Entry(frame, width=25)
    entry_price.grid(row=2, column=1, pady=5)

    btn_frame = tk.Frame(frame, bg="#f4f6f7")
    btn_frame.grid(row=0, column=2, rowspan=4, padx=20)

    ttk.Button(btn_frame, text="Add", width=16, command=add_medicine).grid(row=0, column=0, pady=4)
    ttk.Button(btn_frame, text="Update", width=16, command=update_medicine).grid(row=1, column=0, pady=4)
    ttk.Button(btn_frame, text="Delete", width=16, command=delete_medicine).grid(row=2, column=0, pady=4)
    ttk.Button(btn_frame, text="Sell", width=16, command=sell_medicine).grid(row=3, column=0, pady=4)
    ttk.Button(btn_frame, text="Clear Fields", width=16, command=clear_fields).grid(row=4, column=0, pady=4)

    tree = ttk.Treeview(root, columns=("ID", "Name", "Quantity", "Price"), show='headings')
    tree.heading("ID", text="ID")
    tree.heading("Name", text="Name")
    tree.heading("Quantity", text="Quantity")
    tree.heading("Price", text="Price (₹)")
    tree.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
    tree.bind("<ButtonRelease-1>", on_row_select)

    scrollbar = ttk.Scrollbar(root, orient="vertical", command=tree.yview)
    tree.configure(yscrollcommand=scrollbar.set)
    scrollbar.pack(side="right", fill="y")

    refresh_table()
    root.mainloop()

# --- Admin Login Window ---
def show_login():
    login_win = tk.Tk()
    login_win.title("Admin Login")
    login_win.geometry("300x180")
    login_win.configure(bg="#f4f6f7")

    tk.Label(login_win, text="Admin Login", font=("Segoe UI", 14, "bold"), bg="#f4f6f7", fg="#007acc").pack(pady=10)

    tk.Label(login_win, text="Username:", bg="#f4f6f7").pack()
    user_entry = tk.Entry(login_win)
    user_entry.pack()

    tk.Label(login_win, text="Password:", bg="#f4f6f7").pack()
    pass_entry = tk.Entry(login_win, show="*")
    pass_entry.pack()

    def validate_login():
        u = user_entry.get()
        p = pass_entry.get()
        cursor.execute("SELECT * FROM admin WHERE username=? AND password=?", (u, p))
        if cursor.fetchone():
            login_win.destroy()
            launch_main_dashboard()
        else:
            messagebox.showerror("Login Failed", "Invalid username or password")

    tk.Button(login_win, text="Login", width=15, command=validate_login).pack(pady=10)
    login_win.mainloop()

# Start Application
show_login()